<a href="https://colab.research.google.com/github/minhfua/GOGO-PROJECT/blob/main/GOGO_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok streamlit-folium folium requests polyline scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.5/530.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.7 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import folium
from streamlit_folium import st_folium
import requests
import polyline
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl
from datetime import datetime

# --- 1. HỆ THỐNG MỜ (Giữ nguyên bản gốc của bạn) ---
dist_in = ctrl.Antecedent(np.arange(0, 51, 0.5), 'dist_in')
traf_in = ctrl.Antecedent(np.arange(0, 11, 1), 'traf_in')
time_out = ctrl.Consequent(np.arange(0, 91, 1), 'time_out')
dist_in['gan'] = fuzzy.trimf(dist_in.universe, [0, 0, 5])
dist_in['vua'] = fuzzy.trimf(dist_in.universe, [3, 10, 15])
dist_in['xa'] = fuzzy.trimf(dist_in.universe, [12, 50, 50])
traf_in['thap'] = fuzzy.trimf(traf_in.universe, [0, 0, 5])
traf_in['cao'] = fuzzy.trimf(traf_in.universe, [4, 10, 10])
time_out['nhanh'] = fuzzy.trimf(time_out.universe, [0, 5, 12])
time_out['vua'] = fuzzy.trimf(time_out.universe, [10, 25, 45])
time_out['cham'] = fuzzy.trimf(time_out.universe, [40, 70, 90])
r1 = ctrl.Rule(dist_in['gan'] & traf_in['thap'], time_out['nhanh'])
r2 = ctrl.Rule(dist_in['gan'] & traf_in['cao'], time_out['vua'])
r3 = ctrl.Rule(dist_in['vua'] & traf_in['thap'], time_out['vua'])
r4 = ctrl.Rule(dist_in['vua'] & traf_in['cao'], time_out['cham'])
r5 = ctrl.Rule(dist_in['xa'], time_out['cham'])
t_ctrl = ctrl.ControlSystem([r1, r2, r3, r4, r5])
t_sim = ctrl.ControlSystemSimulation(t_ctrl)

# --- 2. GIAO DIỆN NÂNG CẤP (GLASSMORPHISM & ADAPTIVE) ---
st.set_page_config(page_title="GOGO Smart Ride", layout="wide")

st.markdown("""
    <style>
        :root { --ueh-green: #116938; --eco-green: #2ecc71; }

        /* Đảm bảo màu chữ thích ứng toàn app */
        .stApp { color: var(--text-color); }

        /* Metric Card - Giữ hiệu ứng Glassmorphism nhưng adaptive màu chữ */
        .metric-card {
            background: rgba(128, 128, 128, 0.1);
            backdrop-filter: blur(10px);
            border-radius: 15px; padding: 20px;
            border: 1px solid rgba(128, 128, 128, 0.2);
            box-shadow: 0 8px 32px 0 rgba(31, 38, 135, 0.07);
        }
        .metric-card small { color: var(--text-color); opacity: 0.8; }

        /* Sidebar Adaptive */
        [data-testid="stSidebar"] { border-right: 1px solid rgba(128, 128, 128, 0.1); }
        .sidebar-box {
            background: rgba(128, 128, 128, 0.05);
            padding: 15px; border-radius: 12px;
            border: 1px solid rgba(128, 128, 128, 0.1);
            margin-bottom: 20px;
        }

        /* Giữ nguyên style Button của bạn */
        div.stButton > button {
            background: linear-gradient(90deg, #116938, #27ae60);
            color: white !important; border: none; border-radius: 25px;
            height: 3.5em; font-weight: bold; letter-spacing: 1px;
            box-shadow: 0 4px 15px rgba(17, 105, 56, 0.3); transition: 0.3s;
        }
        div.stButton > button:hover { transform: translateY(-3px); box-shadow: 0 6px 20px rgba(17, 105, 56, 0.4); }

        /* Hiệu ứng xe chạy (Giữ nguyên bản gốc) */
        .status-container {
            background: #1e1e1e; border-radius: 20px; height: 260px;
            position: relative; overflow: hidden; display: flex; justify-content: center; align-items: center;
        }
        .road-marks {
            position: absolute; width: 100%; height: 2px; background: rgba(255,255,255,0.2);
            top: 210px; animation: road-flow 0.3s linear infinite;
        }
        @keyframes road-flow { from { transform: translateX(100%); } to { transform: translateX(-100%); } }
        .bike-float { font-size: 80px; z-index: 10; animation: bike-move 0.5s ease-in-out infinite; }
        @keyframes bike-move { 0%, 100% { transform: translateY(0) rotate(-2deg); } 50% { transform: translateY(-8px) rotate(2deg); } }

        /* Box thông báo trạng thái */
        .status-info-box {
            background: rgba(128, 128, 128, 0.05);
            padding: 20px; border-radius: 15px; margin-top: 15px;
            border-left: 5px solid var(--ueh-green);
        }

        .eco-badge {
            background: rgba(46, 125, 50, 0.15); color: #2ecc71; padding: 5px 15px; border-radius: 20px;
            font-weight: bold; border: 1px solid #2e7d32; display: inline-block;
        }
    </style>
""", unsafe_allow_html=True)

if 'data' not in st.session_state:
    st.session_state.data = None

st.markdown("""
    <div style='text-align: center; padding: 10px;'>
        <h1 style='color: #116938; margin-bottom: 0;'>🌿 GOGO SMART RIDE</h1>
        <p style='opacity: 0.8; font-style: italic;'>Di chuyển thông minh - Bảo vệ hành tinh</p>
    </div>
""", unsafe_allow_html=True)

# --- SIDEBAR ---
with st.sidebar:
    st.markdown("""
        <div style='text-align: center; padding-bottom: 20px;'>
            <div style='background: #116938; width: 60px; height: 60px; line-height: 60px; border-radius: 50%; margin: 0 auto 10px auto; font-size: 30px; color: white;'>
                🏫
            </div>
            <h2 style='color: #116938; font-size: 1.4rem; margin-bottom: 0;'>Chuyến xe xanh GOGO</h2>
            <p style='opacity: 0.7; font-size: 0.8rem; letter-spacing: 1px;'>UEH ECO-SYSTEM</p>
        </div>
    """, unsafe_allow_html=True)

    st.markdown("### 🗺️ Thông tin lộ trình")
    with st.container():
        st.markdown("<div class='sidebar-box'>", unsafe_allow_html=True)
        s_addr = st.text_input("📍 Điểm đón:", "UEH")
        e_addr = st.text_input("🏁 Điểm đến:", "Bệnh viện Chợ Rẫy")
        st.markdown("</div>", unsafe_allow_html=True)

    st.markdown("### 🛵 Loại dịch vụ")
    car_option = st.radio(
        "Chọn xe phù hợp:",
        [
            ('Xe ôm Saver (Xăng)', 3000, 95, "💰"),
            ('Xe ôm Eco 🍃 (Điện)', 5000, 175, "⚡"),
            ('Xe ôm Premium (Xăng)', 8000, 85, "✨")
        ],
        format_func=lambda x: f"{x[3]} {x[0]}"
    )

    st.markdown("### ⚙️ Tuỳ chọn thêm")
    with st.expander("Bảo hiểm & Phụ phí", expanded=True):
        is_env = st.toggle("Phí môi trường (1k)", value=True)
        is_cover = st.toggle("Bảo hiểm Ride Cover", value=False)

    if st.button("🚀 BẮT ĐẦU TÍNH PHÍ"):
        st.session_state.data = "processing"

# --- 3. LOGIC XỬ LÝ (Giữ nguyên 100%) ---
if st.session_state.data == "processing":
    def get_loc(address):
        try:
            res = requests.get(f"https://nominatim.openstreetmap.org/search?format=json&q={address}", headers={'User-Agent': 'RideApp/1.0'}).json()
            return (float(res[0]['lat']), float(res[0]['lon'])) if res else None
        except: return None

    start_c = get_loc(s_addr)
    end_c = get_loc(e_addr)

    if start_c and end_c:
        r_res = requests.get(f"http://router.project-osrm.org/route/v1/driving/{start_c[1]},{start_c[0]};{end_c[1]},{end_c[0]}?overview=full").json()
        km = r_res['routes'][0]['distance'] / 1000
        h = datetime.now().hour
        t_val = 8 if (7 <= h <= 9) or (16 <= h <= 19) else 2

        base_time = km * 4
        try:
            t_sim.input['dist_in'] = km
            t_sim.input['traf_in'] = t_val
            t_sim.compute()
            fuzzy_time = t_sim.output['time_out']
            if km <= 2: final_time = base_time + (1 if t_val < 5 else 3)
            else: final_time = (base_time + fuzzy_time) / 1.8
        except: final_time = km * 5
        if 0.8 <= km <= 1.2 and t_val < 5: final_time = 5

        total = km * car_option[1]
        if t_val > 5: total *= 1.3
        if is_env: total += 1000
        if is_cover: total += 2000
        co2_saved = km * car_option[2]

        st.session_state.data = {
            'price': total, 'km': km, 'time': final_time, 'peak': t_val > 5,
            's': start_c, 'e': end_c, 'geom': r_res['routes'][0]['geometry'],
            'co2': co2_saved,
            'is_eco': "Eco" in car_option[0]
        }

# --- 4. HIỂN THỊ (Hòa quyện Glassmorphism & Adaptive) ---
if isinstance(st.session_state.data, dict):
    d = st.session_state.data
    cols = st.columns(4)
    with cols[0]: st.markdown(f"<div class='metric-card'><small>TỔNG PHÍ</small><h2 style='color:#116938;margin:0;'>{int(d['price']):,}đ</h2></div>", unsafe_allow_html=True)
    with cols[1]: st.markdown(f"<div class='metric-card'><small>QUÃNG ĐƯỜNG</small><h2 style='color:#116938;margin:0;'>{d['km']:.2f}km</h2></div>", unsafe_allow_html=True)
    with cols[2]: st.markdown(f"<div class='metric-card'><small>DỰ KIẾN</small><h2 style='color:#116938;margin:0;'>{round(d['time'])}p</h2></div>", unsafe_allow_html=True)
    with cols[3]:
        co2_color = "#2ecc71" if d['is_eco'] else "#116938"
        st.markdown(f"<div class='metric-card'><small>🍃 CO2 GIẢM</small><h2 style='color:{co2_color};margin:0;'>{int(d['co2'])}g</h2></div>", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    c1, c2 = st.columns([1.5, 1])
    with c1:
        m = folium.Map(location=d['s'], zoom_start=14, tiles='CartoDB positron')
        folium.PolyLine(polyline.decode(d['geom']), color="#116938", weight=8, opacity=0.6).add_to(m)
        folium.Marker(d['s'], icon=folium.Icon(color='green', icon='play', prefix='fa')).add_to(m)
        folium.Marker(d['e'], icon=folium.Icon(color='red', icon='stop', prefix='fa')).add_to(m)
        st_folium(m, width="100%", height=500, key="ride_map")

    with c2:
        bike_icon = "🛵⚡" if d['is_eco'] else "🛵"
        st.markdown(f"<div class='status-container'><div class='road-marks'></div><div class='bike-float'>{bike_icon}</div></div>", unsafe_allow_html=True)
        st.markdown(f"""
            <div class='status-info-box'>
                <p style='margin:0;'><b>Tình trạng:</b> {'🔥 Cao điểm (Giá x1.3)' if d['peak'] else '🟢 Thông thoáng'}</p>
                <p style='margin:0; font-size:0.9em; opacity:0.8;'>Tài xế đang đến đón tại <b>{s_addr}</b>.</p>
                <hr style='opacity:0.2; margin:10px 0;'>
                <div class='eco-badge'>{'Sống xanh vượt trội cùng Eco 🍃' if d['is_eco'] else 'Sống xanh cùng GOGO'}</div>
            </div>
        """, unsafe_allow_html=True)
        if st.button("XÁC NHẬN ĐẶT XE"):
            st.balloons()
            st.success("Yêu cầu đã được gửi! Chúc bạn chuyến đi an toàn.")

Overwriting app.py


In [ ]:
from pyngrok import ngrok
import os
!pkill ngrok
ngrok.kill()
ngrok.set_auth_token("3DCzkYVyk7cztKhBEp7gOn1Yinv_31y5SnP1LeyZkPoADqfK1")
os.system("nohup streamlit run app.py &")
# Đợi 5 giây cho app nổ máy
import time
time.sleep(5)
public_url = ngrok.connect(8501)
print("👉 Bấm vào link này:", public_url)

👉 Bấm vào link này: NgrokTunnel: "https://recharger-deserve-lapel.ngrok-free.dev" -> "http://localhost:8501"
